In [1]:
## simple genai app using langchain
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["LANGCHAIN_API_KEY"]= os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"]= os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_TRACING_V2"]= os.getenv("LANGCHAIN_TRACING_V2")

In [2]:
## data_ingestion 
from langchain_community.document_loaders import WebBaseLoader
import bs4
loader = WebBaseLoader("https://docs.langchain.com/oss/python/deepagents/overview")
loader

C:\Users\preet\AppData\Local\Temp\ipykernel_12960\4091206518.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
c:\Users\preet\OneDrive\Desktop\Genai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
docs=loader.load()

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [5]:
textsplit_docs=text_splitter.split_documents(docs)

In [6]:
textsplit_docs

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/deepagents/overview', 'title': 'Deep Agents overview - Docs by LangChain', 'description': 'Build agents that can plan, use subagents, and leverage file systems for complex tasks', 'language': 'en'}, page_content='Deep Agents overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentFall AMA Series: Live sessions on building, evaluating, deploying, and continously improving agents with LangSmith. Register now →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationDeep Agents overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedQuickstartCustomizationModelsComparison with Claude Agent SDKChangelogDeploymentManaged Deep AgentsBETAGoing to productionExecution environme

In [7]:
from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [8]:
from langchain_community.vectorstores import FAISS
vectordb=FAISS.from_documents(textsplit_docs, embeddings)

In [9]:
vectordb

In [10]:
query="what is deepagents in langchain?"
query_results=vectordb.similarity_search_with_score(query)

In [11]:
query_results

[(Document(id='51424186-fb40-45a3-813c-af6ddd4b2f9a', metadata={'source': 'https://docs.langchain.com/oss/python/deepagents/overview', 'title': 'Deep Agents overview - Docs by LangChain', 'description': 'Build agents that can plan, use subagents, and leverage file systems for complex tasks', 'language': 'en'}, page_content='Deep Agents is an “agent harness”. It is the same core tool calling loop as other agent frameworks, but with built-in capabilities that make agents reliable for real tasks:\nExecution environmentTools, virtual filesystem, optional sandbox, and REPL (interpreter)Context managementSkills, memory, summarization, context offloading, and prompt cachingDelegationSubagent spawning and optional task planningSteeringHuman-in-the-loop approval and interrupts\ndeepagents is a standalone library built on top of LangChain’s core building blocks for agents. It uses the LangGraph runtime for durable execution, streaming, human-in-the-loop, and other features.\nLangChain is the fra

In [13]:
import sys
!{sys.executable} -m pip install -U langchain langchain-core langchain-community

In [12]:
from langchain_ollama import ChatOllama
llm=ChatOllama(model="gemma:2b")

In [13]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_template(
"""
Answer the following question based only on the provided context:
<context>
{context}
</context>

Question: {input}
"""
)
document_chain=create_stuff_documents_chain(llm, prompt)

In [15]:
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2'}}, model='gemma:2b')
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [16]:
pip install -U langchain langchain-classic langchain-community langchain-core

Note: you may need to restart the kernel to use updated packages.


In [17]:
from langchain_core.documents import Document
document_chain.invoke(
    {
        "input": "Deep Agents is the easiest way to start building agents and applications that are powered by LLMs",
        "context":[Document(page_content=" Optional capabilities such as task planning and skills extend the harness when your use case needs them. You can use deep agents for any task, including complex, multi-step tasks.Deep Agents comes with the following capabilities:")]
    }
)


'According to the context, Deep Agents comes with the following capabilities:\n\n* Task planning\n* Skills'

In [19]:
#### retriever
## input->retriever->vectorstoredb->retriever
retriever=vectordb.as_retriever()

In [20]:
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)

In [21]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000024387AA7B60>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n'), additional_kwargs={})])
            | Chat

In [ ]:
response=retrieval_chain.invoke({"input":"what is deepagents?"})
response['answer']